In [1]:
import pandas as pd
import sqlite3

date_cols = ["applied_date", "documents_received_date", "interviewed_date", "offer_made_date",
             "training_completed_date", "induction_attended_date",
             "shadowing_completed_date", "started_date"]
df = pd.read_csv("data/recruitment_clean.csv", parse_dates=date_cols)

con = sqlite3.connect(":memory:")          # create a temporary database
df.to_sql("candidates", con, index=False)  # save our table into it, named "candidates"

def sql(query):
    return pd.read_sql(query, con)         # helper: run SQL, show result as a table

In [2]:
sql("""
SELECT candidate_id, role, source, applied_date
FROM candidates
LIMIT 5
""")

,candidate_id,role,source,applied_date
0,C0001,Healthcare Assistant,Facebook,2025-08-22 00:00:00
1,C0002,Domiciliary Care Worker,Facebook,2026-04-09 00:00:00
2,C0003,Healthcare Assistant,Website,2025-02-19 00:00:00
3,C0004,Domiciliary Care Worker,Facebook,2025-10-27 00:00:00
4,C0005,Healthcare Assistant,Agency,2026-05-04 00:00:00


In [3]:
sql("""
SELECT COUNT(*)            AS applied,
       COUNT(started_date) AS started,
       ROUND(100.0 * COUNT(started_date) / COUNT(*), 1) AS pct_started
FROM candidates
""")

,applied,started,pct_started
0,1200,284,23.7


In [4]:
sql("""
SELECT source,
       COUNT(*)            AS applied,
       COUNT(started_date) AS started,
       ROUND(100.0 * COUNT(started_date) / COUNT(*), 1) AS conversion_pct
FROM candidates
GROUP BY source
ORDER BY conversion_pct DESC
""")

,source,applied,started,conversion_pct
0,Referral,166,52,31.3
1,Agency,124,36,29.0
2,Website,179,50,27.9
3,Indeed,505,120,23.8
4,Facebook,226,26,11.5


In [5]:
sql("""
SELECT role,
       COUNT(*)            AS applied,
       COUNT(started_date) AS started,
       ROUND(100.0 * COUNT(started_date) / COUNT(*), 1) AS conversion_pct
FROM candidates
WHERE role <> 'Unknown'
GROUP BY role
""")

,role,applied,started,conversion_pct
0,Domiciliary Care Worker,638,137,21.5
1,Healthcare Assistant,552,146,26.4


In [6]:
sql("""
SELECT ROUND(AVG(julianday(started_date) - julianday(applied_date)), 1) AS avg_days_to_hire
FROM candidates
WHERE started_date IS NOT NULL
""")

,avg_days_to_hire
0,47.6


In [7]:
sql("""
SELECT CASE
         WHEN julianday(started_date) - julianday(applied_date) <= 30 THEN '0-30 days'
         WHEN julianday(started_date) - julianday(applied_date) <= 60 THEN '31-60 days'
         ELSE '60+ days'
       END AS hire_speed,
       COUNT(*) AS starters
FROM candidates
WHERE started_date IS NOT NULL
GROUP BY hire_speed
ORDER BY hire_speed
""")

,hire_speed,starters
0,0-30 days,49
1,31-60 days,174
2,60+ days,61


## SQL cross-check
All headline figures (overall conversion, conversion by source, average time-to-hire) were recalculated in SQL and match the pandas results exactly. Additional finding: Healthcare Assistants convert at 26.4% vs 21.5% for Domiciliary Care Workers.